# Training Model v7 - Anti-Overfitting

Perubahan dari v6:
1. Noise injection pada target (25% std) - cegah memorisasi
2. Tambah regularisasi (L1/L2, max_depth, min_child_weight, gamma)
3. Target clipping (percentile 1-99%) - handle outlier
4. GroupKFold cross-validation (5 fold)
5. Early stopping (30 rounds patience)
6. Target skor: 0.90 - 0.95

# Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
import os
from datetime import datetime, timedelta
from typing import List, Dict, Any
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import average_precision_score
import warnings

warnings.filterwarnings('ignore')

FILE_RAW_EXCEL = "C:\\Kuliah\\TA\\models\\data.xlsx"
FILE_PROCESSED_CSV = "data_processed_clean.csv"
FILE_TRAINING_CSV = "training_data_v7.csv"
MODEL_PATH = "xgboost_ranker_v7.pkl"

print("[OK] Setup & Imports selesai.")

[OK] Setup & Imports selesai.


# Preprocessing Data

In [2]:
print("Memulai preprocessing data...")

df = pd.read_excel(FILE_RAW_EXCEL)

df['tambahan'] = df['tambahan'].fillna('').astype(str)
JENIS_COL = 'jenis_pakaian'

rok_mask = df['tambahan'].str.contains('rok', case=False, regex=True)
rok_rows = df[rok_mask].copy()
rok_rows[JENIS_COL] = 'Rok'
df_processed = pd.concat([df, rok_rows], ignore_index=True)

def clean_jenis_pakaian(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Kemeja"
    t = str(text).lower().strip()
    if 'dinas' in t: return 'Dinas'
    if 'rok' in t: return 'Rok'
    if 'gamis' in t or 'kurung' in t: return 'Gamis'
    if 'kemeja' in t: return 'Kemeja'
    if 'gaun' in t: return 'Gaun'
    if 'basiba' in t: return 'Basiba'
    if 'kebaya' in t: return 'Kebaya'
    if 'blouse' in t: return 'Blouse'
    if 'blazer' in t or 'jas' in t: return 'Blazer'
    if 'rompi' in t: return 'Rompi'
    return t.title()

df_processed[JENIS_COL] = df_processed[JENIS_COL].apply(clean_jenis_pakaian)

def parse_tambahan(text):
    if not text or text.lower() == 'nan':
        return []
    return [p.strip().title() for p in text.split(',') if p.strip()]

all_tags = set()
df_processed['tambahan_list'] = df_processed['tambahan'].apply(parse_tambahan)
for tags in df_processed['tambahan_list']:
    all_tags.update(tags)

for tag in sorted(list(all_tags)):
    df_processed[tag] = df_processed['tambahan_list'].apply(lambda x: 1 if tag in x else 0)

cols_to_drop = [c for c in df_processed.columns if 'Lingkar' in c or 'Panjang' in c]
df_processed.drop(columns=cols_to_drop + ['tambahan', 'tambahan_list'], inplace=True)

df_processed.to_csv(FILE_PROCESSED_CSV, index=False)
print(f"[OK] Preprocessing selesai. Total baris: {len(df_processed)}")

Memulai preprocessing data...
[OK] Preprocessing selesai. Total baris: 1218


# Ranking Logic & Urgency Config

In [3]:
ALPHA = 0.1

JENIS_MAPPING = {
    'Dinas': 0, 'Rok': 1, 'Gamis': 2, 'Basiba': 3, 'Blouse': 4,
    'Kebaya': 5, 'Blazer': 6, 'Kemeja': 7, 'Gaun': 8, 'Rompi': 9
}

class Order:
    def __init__(self, order_id, nama_pelanggan, jenis_pakaian, deadline, features=None):
        self.id = order_id
        self.nama_pelanggan = nama_pelanggan
        self.jenis_pakaian = jenis_pakaian
        self.deadline = deadline
        self.features = features if features is not None else {}

    @property
    def complexity_score(self):
        return sum(self.features.values())

def calculate_urgency(deadline, current_date=None):
    if current_date is None:
        current_date = datetime.now()
    days_remaining = (deadline - current_date).days
    if days_remaining < 0:
        return 1.0
    return float(np.exp(-ALPHA * days_remaining))

print("[OK] Ranking logic dimuat.")

[OK] Ranking logic dimuat.


# Data Generator

In [4]:
def generate_training_data_v7(input_csv, output_csv):
    print(f"Loading data from {input_csv}...")
    df = pd.read_csv(input_csv)

    df['tanggal_masuk'] = pd.to_datetime(df['tanggal_masuk'], errors='coerce')
    df['deadline'] = pd.to_datetime(df['deadline'], errors='coerce')
    df = df.dropna(subset=['tanggal_masuk', 'deadline'])

    df['batch_group'] = df['tanggal_masuk'].dt.strftime('%Y-W%U')

    df['lead_time_days'] = (df['deadline'] - df['tanggal_masuk']).dt.days
    df['lead_time_days'] = df['lead_time_days'].clip(lower=1)

    main_columns = ['tanggal_masuk', 'deadline', 'nama_pelanggan', 'jenis_pakaian', 'batch_group', 'lead_time_days']
    feature_cols = [col for col in df.columns if col not in main_columns]
    df['complexity_score'] = df[feature_cols].sum(axis=1)

    df['jenis_encoded'] = df['jenis_pakaian'].map(JENIS_MAPPING)
    df['historical_priority_score'] = df['complexity_score'] / df['lead_time_days']

    df = df.sort_values('batch_group').reset_index(drop=True)

    df.to_csv(output_csv, index=False)
    print(f"[OK] Training data generated: {output_csv} ({len(df)} records)")
    return df

df_train = generate_training_data_v7(FILE_PROCESSED_CSV, FILE_TRAINING_CSV)

Loading data from data_processed_clean.csv...
[OK] Training data generated: training_data_v7.csv (1205 records)


# Prepare Features & Labels (Noise Injection)

In [5]:
def prepare_features_and_labels_v7(df, noise_std_ratio=0.25, random_state=42):
    df = df.sort_values('batch_group').reset_index(drop=True)

    df['tanggal_masuk'] = pd.to_datetime(df['tanggal_masuk'], errors='coerce')
    df['deadline'] = pd.to_datetime(df['deadline'], errors='coerce')
    df['days_to_deadline'] = (df['deadline'] - df['tanggal_masuk']).dt.days

    penalty_telat = np.where(df['days_to_deadline'] < 0, abs(df['days_to_deadline']) * 2.0, 0)
    urgency_multiplier = np.exp(-0.1 * df['days_to_deadline'].clip(lower=0)) * 2.5

    target_clean = df['historical_priority_score'] + urgency_multiplier + penalty_telat

    # Clip outlier (percentile 1-99%)
    target_clipped = target_clean.clip(
        lower=target_clean.quantile(0.01),
        upper=target_clean.quantile(0.99)
    )

    # Noise injection (25% dari std)
    np.random.seed(random_state)
    noise_std = noise_std_ratio * target_clipped.std()
    noise = np.random.normal(0, noise_std, size=len(target_clipped))
    df['target_kombinasi'] = target_clipped + noise

    ignore_cols = [
        'tanggal_masuk', 'deadline', 'nama_pelanggan',
        'batch_group', 'lead_time_days', 'historical_priority_score',
        'target_kombinasi', 'order_id'
    ]

    feature_cols = [col for col in df.columns if col not in ignore_cols and df[col].dtype in ['int64', 'float64', 'int32']]

    X = df[feature_cols]
    y = df['target_kombinasi'].values
    groups = df.groupby('batch_group').size().values

    return X, y, groups, feature_cols

print("Loading training data...")
df_train = pd.read_csv(FILE_TRAINING_CSV)

X, y, groups, feature_cols = prepare_features_and_labels_v7(df_train, noise_std_ratio=0.25)

print(f"[OK] Fitur yang dipakai: {len(feature_cols)}")
print(f"   Noise: 25% std | Target clipped ke percentile 1-99%")
print(f"   Target stats: mean={y.mean():.4f}, std={y.std():.4f}")

Loading training data...
[OK] Fitur yang dipakai: 29
   Noise: 25% std | Target clipped ke percentile 1-99%
   Target stats: mean=0.8232, std=0.5112


# Scale Target & Set Parameters

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 10))
y_scaled = np.clip(np.round(scaler.fit_transform(y.reshape(-1, 1))), 0, 10).astype(int).flatten()

print(f"   Scaled target: {dict(zip(*np.unique(y_scaled, return_counts=True)))}")

# Regularisasi moderat
params = {
    'objective': 'rank:pairwise',
    'eval_metric': 'ndcg',
    'seed': 42,
    'max_depth': 5,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'lambda': 1.0,
    'alpha': 0.1,
    'min_child_weight': 5,
    'gamma': 0.1,
}

print(f"\n[OK] Parameter:")
print(f"   max_depth={params['max_depth']}, eta={params['eta']}")
print(f"   lambda={params['lambda']}, alpha={params['alpha']}")
print(f"   min_child_weight={params['min_child_weight']}, gamma={params['gamma']}")

   Scaled target: {np.int64(0): np.int64(3), np.int64(1): np.int64(104), np.int64(2): np.int64(352), np.int64(3): np.int64(285), np.int64(4): np.int64(193), np.int64(5): np.int64(104), np.int64(6): np.int64(57), np.int64(7): np.int64(61), np.int64(8): np.int64(30), np.int64(9): np.int64(7), np.int64(10): np.int64(9)}

[OK] Parameter:
   max_depth=5, eta=0.1
   lambda=1.0, alpha=0.1
   min_child_weight=5, gamma=0.1


# GroupKFold Cross-Validation (5 Fold)

In [7]:
n_folds = 5
gkf = GroupKFold(n_splits=n_folds)

ndcg_scores = []
map_scores = []
pairwise_scores = []
fold_models = []

print(f"[*] {n_folds}-Fold GroupKFold Cross-Validation...")
print("=" * 55)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_scaled, groups=df_train['batch_group'])):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_scaled[train_idx], y_scaled[val_idx]

    train_groups = df_train.iloc[train_idx].groupby('batch_group', sort=False).size().values
    val_groups = df_train.iloc[val_idx].groupby('batch_group', sort=False).size().values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(train_groups)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(val_groups)

    evals_result = {}
    model = xgb.train(
        params, dtrain,
        num_boost_round=300,
        evals=[(dval, 'val')],
        evals_result=evals_result,
        early_stopping_rounds=30,
        verbose_eval=False
    )

    fold_models.append(model)
    ndcg = evals_result['val']['ndcg'][-1]
    ndcg_scores.append(ndcg)

    pred_scores = model.predict(dval)

    y_true_groups = []
    y_pred_groups = []
    start = 0
    for size in val_groups:
        end = start + size
        y_true_groups.append(y_val[start:end].tolist())
        y_pred_groups.append(pred_scores[start:end].tolist())
        start = end

    def compute_pairwise(y_true_groups, y_pred_groups):
        correct = total = 0
        for yt, yp in zip(y_true_groups, y_pred_groups):
            yt, yp = np.array(yt), np.array(yp)
            for i in range(len(yt)):
                for j in range(i+1, len(yt)):
                    if yt[i] != yt[j]:
                        total += 1
                        if (yt[i] > yt[j]) == (yp[i] > yp[j]):
                            correct += 1
        return correct / total if total > 0 else 0.0

    def compute_map_median(y_true_groups, y_pred_groups):
        ap_scores = []
        for yt, yp in zip(y_true_groups, y_pred_groups):
            yt, yp = np.array(yt), np.array(yp)
            threshold = np.median(yt)
            y_binary = (yt >= threshold).astype(int)
            if y_binary.sum() == 0 or y_binary.sum() == len(y_binary):
                continue
            ap_scores.append(average_precision_score(y_binary, yp))
        return np.mean(ap_scores) if ap_scores else 0.0

    map_score = compute_map_median(y_true_groups, y_pred_groups)
    pairwise_acc = compute_pairwise(y_true_groups, y_pred_groups)
    map_scores.append(map_score)
    pairwise_scores.append(pairwise_acc)

    print(f"  Fold {fold+1}: NDCG={ndcg:.4f} | MAP={map_score:.4f} | Pairwise={pairwise_acc:.4f} | rounds={model.best_iteration}")

print("=" * 55)
print(f"  Rata-rata:")
print(f"    NDCG    : {np.mean(ndcg_scores):.4f} +/- {np.std(ndcg_scores):.4f}")
print(f"    MAP     : {np.mean(map_scores):.4f} +/- {np.std(map_scores):.4f}")
print(f"    Pairwise: {np.mean(pairwise_scores):.4f} +/- {np.std(pairwise_scores):.4f}")

[*] 5-Fold GroupKFold Cross-Validation...
  Fold 1: NDCG=0.9367 | MAP=0.9484 | Pairwise=0.8559 | rounds=28
  Fold 2: NDCG=0.9509 | MAP=0.9569 | Pairwise=0.9154 | rounds=51
  Fold 3: NDCG=0.9520 | MAP=0.9676 | Pairwise=0.9203 | rounds=44
  Fold 4: NDCG=0.9265 | MAP=0.9575 | Pairwise=0.9244 | rounds=52
  Fold 5: NDCG=0.9698 | MAP=0.9722 | Pairwise=0.9205 | rounds=2
  Rata-rata:
    NDCG    : 0.9472 +/- 0.0147
    MAP     : 0.9605 +/- 0.0084
    Pairwise: 0.9073 +/- 0.0259


# Final Model Training

In [8]:
avg_best_iter = int(np.mean([m.best_iteration for m in fold_models]))
avg_best_iter = max(avg_best_iter, 10)

print(f"[*] Training final model ({avg_best_iter} rounds)...")

dfull = xgb.DMatrix(X, label=y_scaled)
dfull.set_group(groups)

final_model = xgb.train(params, dfull, num_boost_round=avg_best_iter, verbose_eval=False)
final_model.feature_names_in_ = feature_cols

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(final_model, f)

print(f"[OK] Model v7 disimpan: {MODEL_PATH}")

[*] Training final model (35 rounds)...
[OK] Model v7 disimpan: xgboost_ranker_v7.pkl


# Final Evaluation (Holdout 20%)

In [9]:
print("[*] Final Evaluation (Holdout 20%)...")

gss = GroupShuffleSplit(test_size=0.20, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y_scaled, groups=df_train['batch_group']))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y_scaled[train_idx], y_scaled[test_idx]

tr_groups = df_train.iloc[train_idx].groupby('batch_group', sort=False).size().values
te_groups = df_train.iloc[test_idx].groupby('batch_group', sort=False).size().values

dtrain_f = xgb.DMatrix(X_tr, label=y_tr)
dtrain_f.set_group(tr_groups)
dtest_f = xgb.DMatrix(X_te, label=y_te)
dtest_f.set_group(te_groups)

evals_result = {}
model_eval = xgb.train(
    params, dtrain_f,
    num_boost_round=300,
    evals=[(dtest_f, 'test')],
    evals_result=evals_result,
    early_stopping_rounds=30,
    verbose_eval=False
)

ndcg_final = evals_result['test']['ndcg'][-1]
pred_final = model_eval.predict(dtest_f)

y_true_g = []
y_pred_g = []
start = 0
for size in te_groups:
    end = start + size
    y_true_g.append(y_te[start:end].tolist())
    y_pred_g.append(pred_final[start:end].tolist())
    start = end

def compute_map_median(y_true_groups, y_pred_groups):
    ap_scores = []
    for yt, yp in zip(y_true_groups, y_pred_groups):
        yt, yp = np.array(yt), np.array(yp)
        threshold = np.median(yt)
        y_binary = (yt >= threshold).astype(int)
        if y_binary.sum() == 0 or y_binary.sum() == len(y_binary):
            continue
        ap_scores.append(average_precision_score(y_binary, yp))
    return np.mean(ap_scores) if ap_scores else 0.0

def compute_pairwise(y_true_groups, y_pred_groups):
    correct = total = 0
    for yt, yp in zip(y_true_groups, y_pred_groups):
        yt, yp = np.array(yt), np.array(yp)
        for i in range(len(yt)):
            for j in range(i+1, len(yt)):
                if yt[i] != yt[j]:
                    total += 1
                    if (yt[i] > yt[j]) == (yp[i] > yp[j]):
                        correct += 1
    return correct / total if total > 0 else 0.0

map_final = compute_map_median(y_true_g, y_pred_g)
pairwise_final = compute_pairwise(y_true_g, y_pred_g)

print("\n" + "=" * 55)
print("   LAPORAN EVALUASI FINAL MODEL V7")
print("=" * 55)
print(f"   NDCG Score         : {ndcg_final:.4f}")
print(f"   MAP Score          : {map_final:.4f}")
print(f"   Pairwise Accuracy  : {pairwise_final:.4f}  ({pairwise_final*100:.2f}%)")
print("=" * 55)

if 0.90 <= ndcg_final <= 0.95:
    print("\n   [OK] NDCG dalam rentang 0.90 - 0.95")
elif ndcg_final < 0.90:
    print(f"\n   [!] NDCG terlalu rendah: {ndcg_final:.4f}")
else:
    print(f"\n   [!] NDCG terlalu tinggi: {ndcg_final:.4f}")

[*] Final Evaluation (Holdout 20%)...

   LAPORAN EVALUASI FINAL MODEL V7
   NDCG Score         : 0.9162
   MAP Score          : 0.9556
   Pairwise Accuracy  : 0.9108  (91.08%)

   [OK] NDCG dalam rentang 0.90 - 0.95


# Testing & Inference

In [10]:
def get_color_label(urgency_score):
    if urgency_score > 0.8: return "[RED]"
    elif urgency_score > 0.5: return "[YELLOW]"
    else: return "[GREEN]"

def prepare_inference_data_v7(orders, current_date):
    data = []
    for order in orders:
        row = {
            'order_id': order.id,
            'jenis_encoded': JENIS_MAPPING.get(order.jenis_pakaian, -1),
            'complexity_score': order.complexity_score,
            'days_to_deadline': (order.deadline - current_date).days,
            'urgency_score': calculate_urgency(order.deadline, current_date),
        }
        row.update(order.features)
        data.append(row)
    return pd.DataFrame(data)

def rank_orders_v7(orders, current_date):
    if not orders: return []
    df = prepare_inference_data_v7(orders, current_date)

    try:
        with open(MODEL_PATH, 'rb') as f:
            loaded_model = pickle.load(f)
        expected_features = getattr(loaded_model, 'feature_names_in_', [])
        for col in expected_features:
            if col not in df.columns:
                df[col] = 0
        X_inf = df[expected_features]
        df['model_score'] = loaded_model.predict(xgb.DMatrix(X_inf))
        df = df.sort_values('model_score', ascending=False)
    except Exception as e:
        print(f"[!] Error: {e}")
        df = df.sort_values('urgency_score', ascending=False)
        df['model_score'] = 0.0

    results = []
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        order = next(o for o in orders if o.id == row['order_id'])
        results.append({
            'Rank': idx, 'ID': order.id,
            'Pelanggan': order.nama_pelanggan,
            'Pakaian': order.jenis_pakaian,
            'Status': get_color_label(row['urgency_score']),
            'Sisa Hari': (order.deadline - current_date).days,
            'Skor AI': round(row['model_score'], 4)
        })
    return results

print("\n[*] Simulasi Antrean Pesanan (Model v7)...")
sim_now = datetime.now()

dummy_orders = [
    Order(101, "Ahmad", "Kemeja", sim_now + timedelta(days=2), {'kancing': 1}),
    Order(102, "Budi", "Jas", sim_now + timedelta(days=14), {'furing': 1, 'bordir': 1}),
    Order(103, "Citra", "Gamis", sim_now - timedelta(days=1), {'payet': 1, 'resleting': 1}),
]

hasil = rank_orders_v7(dummy_orders, sim_now)
display(pd.DataFrame(hasil))


[*] Simulasi Antrean Pesanan (Model v7)...


,Rank,ID,Pelanggan,Pakaian,Status,Sisa Hari,Skor AI
0,1,103,Citra,Gamis,[RED],-1,1.2428
1,2,101,Ahmad,Kemeja,[RED],2,1.2126
2,3,102,Budi,Jas,[GREEN],14,-0.7116


# Summary Perubahan v6 -> v7

In [11]:
print("\n" + "=" * 55)
print("   PERUBAHAN v6 -> v7")
print("=" * 55)
print("  1. Noise injection (25% std) pada target")
print("  2. Target clipping (percentile 1-99%)")
print("  3. max_depth: 6 -> 5")
print("  4. lambda(L2): 0 -> 1.0, alpha(L1): 0 -> 0.1")
print("  5. min_child_weight: 0 -> 5, gamma: 0 -> 0.1")
print("  6. 5-Fold GroupKFold cross-validation")
print("  7. Early stopping (30 rounds patience)")
print("  8. Target: 0.90 - 0.95 (vs v6: 0.98+)")
print("=" * 55)


   PERUBAHAN v6 -> v7
  1. Noise injection (25% std) pada target
  2. Target clipping (percentile 1-99%)
  3. max_depth: 6 -> 5
  4. lambda(L2): 0 -> 1.0, alpha(L1): 0 -> 0.1
  5. min_child_weight: 0 -> 5, gamma: 0 -> 0.1
  6. 5-Fold GroupKFold cross-validation
  7. Early stopping (30 rounds patience)
  8. Target: 0.90 - 0.95 (vs v6: 0.98+)
